In [8]:
#输出dict文件的最长的一个单词的长度，并以此作为”定长数组储存词典” 方法的每个单词长度
#统计文档中的单词总数
#统计文档中的字符总数
def find_longest_word_length(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        words = file.read().strip().split()  
        longest_word = max(words, key=len)  
        return len(longest_word) 

def count_total_characters(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read().replace(" ", "")  # 去除空格后统计字符数
        return len(content)

def count_total_words(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        words = file.read().strip().split()
        return len(words)

if __name__ == "__main__":
    file_path = r"D:\code\natural_language_processing\lab5\dict.txt"  
    longest_length = find_longest_word_length(file_path)
    total_characters = count_total_characters(file_path)
    total_words = count_total_words(file_path)

    print(f"最长单词的字符长度是: {longest_length}")
    print(f"文档总字符数是: {total_characters}")
    print(f"文档中共有单词数: {total_words}")

最长单词的字符长度是: 22
文档总字符数是: 654452
文档中共有单词数: 87980


由上以上代码的输出结果，假设用储存每个单词的文档频率和指向倒排记录表的指针的空间大小都是4B
每个单词需要22B的空间来储存
可以计算出采用顶常熟组储存的空间开销为：
（22+4+4）×87980 = 2639400B

In [18]:
#将词典压缩成单一字符串
def compress_dictionary(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read().strip()
    
    words = content.split()
    compressed = ''.join(f"{len(word)}{word}" for word in words)
    
    return compressed


if __name__ == "__main__":
    file_path = r"D:\code\natural_language_processing\lab5\dict.txt"
    result = compress_dictionary(file_path)  
    output_path = r"D:\code\natural_language_processing\lab5\compressed_dict.txt"
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(result)

In [28]:
import time
import re
import os

def is_valid_lowercase(s):
    return bool(re.fullmatch(r'[a-z]+', s))

def longest_common_prefix(words):
    if not words:
        return ''
    min_word = min(words)
    max_word = max(words)
    i = 0
    while i < len(min_word) and min_word[i] == max_word[i]:
        i += 1
    return min_word[:i]

# 新增函数：按局部公共前缀对单词进行分组
def group_by_prefix(words):
    groups = []
    i = 0
    n = len(words)

    while i < n:
        j = i + 1
        prefix = words[i]
        # 查找最大可能的公共前缀组
        while j < n:
            current_prefix = longest_common_prefix([prefix, words[j]])
            if not current_prefix or len(current_prefix) < 1:  # 可设置最小长度如2
                break
            prefix = current_prefix
            j += 1
        groups.append((prefix, words[i:j]))
        i = j
    return groups

# 替换原来的 compress_block 函数
def compress_block(block_words):
    block_words = sorted(block_words)  # 确保按字典序排列
    compressed = ""
    prefixes_in_block = []

    groups = group_by_prefix(block_words)

    for prefix, group in groups:
        if len(group) > 1:
            # 多个词共享前缀
            compressed += f"{len(prefix)}{prefix}*"
            prefixes_in_block.append(prefix)
            for word in group:
                suffix = word[len(prefix):]
                compressed += f"{len(suffix)}${suffix}"
        else:
            # 只有一个词，自己作为前缀
            word = group[0]
            compressed += f"{len(word)}{word}*"
            prefixes_in_block.append(word)

    return compressed, prefixes_in_block

def build_index(compressed_str, k):
    words = re.findall(r'(\d+)([a-z]+)', compressed_str)
    words = [w[1] for w in words]
    index_positions = []
    result_blocks = []
    prefixes_per_block = []

    pos = 0
    for i in range(0, len(words), k):
        block_words = words[i:i + k]
        start_pos = pos
        index_positions.append(start_pos)

        block_str, prefixes = compress_block(block_words)
        result_blocks.append(block_str)
        prefixes_per_block.append(prefixes)

        pos += len(block_str)

    full_compressed = ''.join(result_blocks)
    return full_compressed, index_positions, len(index_positions)

def binary_search_query(compressed_str, index_positions, query_word):
    left, right = 0, len(index_positions) - 1
    found = False

    while left <= right:
        mid = (left + right) // 2
        start = index_positions[mid]
        end = index_positions[mid + 1] if mid + 1 < len(index_positions) else len(compressed_str)
        block_str = compressed_str[start:end]

        entries = re.findall(r'(\d+)(\$|\*)?([a-z]*)', block_str)

        current_prefix = None
        for length, flag, word in entries:
            if flag == '*':
                current_prefix = word
            elif flag == '$':
                if current_prefix is None:
                    continue  # 防止 None + str 错误
                suffix_len = int(length)
                suffix = word[:suffix_len]
                candidate = current_prefix + suffix
                if query_word == candidate:
                    found = True
                    break
            elif word:
                candidate = word
                if query_word == candidate:
                    found = True
                    break
        if found:
            break
        elif mid + 1 < len(index_positions) and query_word > block_str:
            left = mid + 1
        else:
            right = mid - 1

    return found

def main():
    try:
        k = int(input("请输入块大小 k（1~87980）: "))
        if not (1 <= k <= 87980):
            print("错误：k 必须在 1 到 87980 之间")
            return

        query_word = input("请输入要查询的单词（仅限小写字母）: ").strip()
        if not is_valid_lowercase(query_word):
            print("错误：查询词必须为小写字母组成")
            return

        input_path = r"D:\code\natural_language_processing\lab5\compressed_dict.txt"
        output_path = fr"D:\code\natural_language_processing\lab5\compressed_dict_k={k}.txt"

        with open(input_path, 'r', encoding='utf-8') as f:
            compressed_str = f.read().strip()

        

        # 构建压缩后的字符串和索引数组
        full_compressed, index_positions, blocknumber = build_index(compressed_str, k)

        # 写入新文件
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(full_compressed)

        print(f"压缩后文件已保存至：{output_path}")

        # ✅ 查询时间计算（单位：毫秒）
        iterations = 1000
        start_time = time.perf_counter()  # 更高精度计时器
        for _ in range(iterations):
            binary_search_query(full_compressed, index_positions, query_word)
        end_time = time.perf_counter()

        avg_time_ms = ((end_time - start_time) * 1000) / iterations  # 转为毫秒/次
        print(f"平均查询时间为：{avg_time_ms:.4f} 毫秒")  # 显示4位小数

        # 压缩率计算
        new_length = len(full_compressed)
        overhead = (87980 + blocknumber) * 4
        original_size = 2639400  # 假设原始数据大小（单位：B）

        compression_rate = (new_length + overhead) / original_size * 100
        print(f"压缩率为：{compression_rate:.2f}%")

    except Exception as e:
        print(f"发生错误：{e}")

if __name__ == "__main__":
    main()

压缩后文件已保存至：D:\code\natural_language_processing\lab5\compressed_dict_k=100.txt
平均查询时间为：0.1886 毫秒
压缩率为：38.80%
